# PI1M Pretrain -> Finetune (TransCVAE_PSMILES, scalar conductivity 유지)

- 대상: `TransCVAE_PSMILES`의 `CVAE + PriorNet`
- PI1M에는 전도도 라벨이 없으므로 pretrain 단계에서는 **정규화된 scalar property를 상수(기본 0.0)** 로 넣고 reconstruction 중심으로 사전학습
- finetune 단계에서 기존 데이터셋(`utils.dataloader`)의 **scalar conductivity 조건**으로 다시 학습
- 최종 저장: `model_weights_trans_pi1m_finetuned.pth`, `model_weights_prior_trans_pi1m_finetuned.pth`


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import time
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm.auto import tqdm

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def _is_models_root(p: Path) -> bool:
    return (p / "utils").is_dir() and (p / "models").is_dir() and (p / "notebooks").is_dir() and (p / "data").is_dir()


def resolve_project_root():
    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if _is_models_root(p):
            return p
        cand = p / "MY_PAPER_RELATED" / "MODELS"
        if _is_models_root(cand):
            return cand
    raise FileNotFoundError("Could not locate MODELS project root")
PROJECT_ROOT = resolve_project_root()
os.environ["MODELS_VARIANT"] = "LSTM_CVAE"
WORKSPACE_ROOT = PROJECT_ROOT.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_ROOT =', PROJECT_ROOT)
print('WORKSPACE_ROOT =', WORKSPACE_ROOT)
print('device =', device)


In [ ]:
from utils.utils import *
from utils.dataloader import dataset as ft_dataset, train_dataloader, val_dataloader, eval_dataloader
from utils.pi1m_pretrain import build_pi1m_pretrain_loader
from utils.loss_fn import ConditionalVAELoss
from models.LSTM import CVAE, PriorNet

vocab_ft = ft_dataset.vocab
index_to_token_ft = {idx: tok for tok, idx in vocab_ft.items()}
print('Finetune dataset size:', len(ft_dataset))
print('Finetune vocab size:', ft_dataset.vocab_size)
print('Finetune max_len:', ft_dataset.max_len)
print('Property mean/std (normalized space):', float(ft_dataset.properties.mean()), float(ft_dataset.properties.std()))


In [ ]:
# ==== Config ====
LATENT_DIM = 128
PI1M_USE_V2 = True
PI1M_MAX_ROWS = None          # 첫 실행은 100_000 ~ 300_000 권장
PI1M_BATCH_SIZE = 256

PRETRAIN_EPOCHS = 1
PRETRAIN_LR = 3e-4
PRETRAIN_KL_BETA = 0.01
PRETRAIN_FREEZE_PREFIXES = (
    "input_embedding",   # PI1M에는 scalar 라벨이 없으므로 조건 경로 학습 제외
    "to_prop",
    "to_prop_z",
)
PRETRAIN_TRANSFER_SCOPE = "grammar_only"   # "grammar_only" | "all"

FINETUNE_EPOCHS = 200
FINETUNE_STAGE1_EPOCHS = 40    # warmup: backbone freeze, conditional/latent path + prior 적응
FINETUNE_STAGE2_EPOCHS = FINETUNE_EPOCHS - FINETUNE_STAGE1_EPOCHS
assert FINETUNE_STAGE2_EPOCHS >= 0
FINETUNE_LR = 3e-4
FINETUNE_LR_PRIOR = FINETUNE_LR * 0.1
WEIGHT_DECAY = 0.0

RUN_FINETUNE = True
LOAD_EXISTING_PRETRAIN = False

PI1M_CSV = WORKSPACE_ROOT / 'PI1M' / ('PI1M_v2.csv' if PI1M_USE_V2 else 'PI1M.csv')
assert PI1M_CSV.exists(), f'PI1M CSV not found: {PI1M_CSV}'

WEIGHTS_DIR = PROJECT_ROOT / 'checkpoints'
PRETRAIN_DIR = WEIGHTS_DIR / 'pretrain'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)

print('PI1M_CSV =', PI1M_CSV)
print('LATENT_DIM =', LATENT_DIM)
print('PRETRAIN_EPOCHS =', PRETRAIN_EPOCHS, 'FINETUNE_EPOCHS =', FINETUNE_EPOCHS)
print('PRETRAIN_TRANSFER_SCOPE =', PRETRAIN_TRANSFER_SCOPE)
print('PRETRAIN_FREEZE_PREFIXES =', PRETRAIN_FREEZE_PREFIXES)
print('FINETUNE_STAGE1_EPOCHS =', FINETUNE_STAGE1_EPOCHS, 'FINETUNE_STAGE2_EPOCHS =', FINETUNE_STAGE2_EPOCHS)


In [ ]:
# ==== Build PI1M pretrain dataloader ====
pi1m_ds, pi1m_loader = build_pi1m_pretrain_loader(
    csv_path=PI1M_CSV,
    batch_size=PI1M_BATCH_SIZE,
    cache_dir=PROJECT_ROOT / 'checkpoints' / 'cache',
    smiles_col='SMILES',
    max_rows=PI1M_MAX_ROWS,
    shuffle=True,
)
print('PI1M rows read:', getattr(pi1m_ds, 'num_rows_read', 'n/a'))
print('PI1M valid rows:', getattr(pi1m_ds, 'num_rows_valid', len(pi1m_ds)))
print('PI1M vocab size:', pi1m_ds.vocab_size)
print('PI1M max_len:', pi1m_ds.max_len)
print('PI1M property placeholder:', float(pi1m_ds.properties[0].view(-1)[0].item()))



In [ ]:
# ==== Helpers ====
def _construct_with_optional_kwargs(cls, **kwargs):
    import inspect
    sig = inspect.signature(cls.__init__)
    allowed = {k: v for k, v in kwargs.items() if k in sig.parameters and v is not None}
    return cls(**allowed)

def build_cvae(vocab_size=None, max_len=None, pad_idx=None, latent_dim=LATENT_DIM):
    model = _construct_with_optional_kwargs(
        CVAE,
        latent_dim=latent_dim,
        vocab_size=vocab_size,
        max_len=max_len,
        pad_idx=pad_idx,
    )
    return model.to(device)

def build_prior(max_len=None, latent_dim=LATENT_DIM):
    prior = _construct_with_optional_kwargs(PriorNet, y_dim=1, latent_dim=latent_dim, max_len=max_len)
    return prior.to(device)

def cvae_recon_kl_pretrain_loss(logits, target_tokens, mu_q, lv_q, pad_idx, beta=0.01):
    B, _, V = logits.shape
    recon = F.cross_entropy(
        logits.reshape(-1, V),
        target_tokens.reshape(-1),
        ignore_index=pad_idx,
        reduction='sum',
    ) / B
    kld = -0.5 * (1.0 + lv_q - mu_q.pow(2) - lv_q.exp())
    kld = kld.sum(-1).mean()
    loss = recon + beta * kld
    return loss, recon.detach(), kld.detach()

def _count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

def set_requires_grad_by_prefix(module, freeze_prefixes=(), unfreeze_all=False):
    if unfreeze_all:
        for p in module.parameters():
            p.requires_grad = True
        return
    for name, p in module.named_parameters():
        if any(name.startswith(pref) for pref in freeze_prefixes):
            p.requires_grad = False
        else:
            p.requires_grad = True

def pretrain_freeze_non_target_heads(model, freeze_prefixes=PRETRAIN_FREEZE_PREFIXES):
    set_requires_grad_by_prefix(model, freeze_prefixes=freeze_prefixes, unfreeze_all=False)
    total, trainable = _count_params(model)
    print(f'[Pretrain] model trainable params: {trainable:,} / {total:,}')
    frozen_examples = [n for n, p in model.named_parameters() if not p.requires_grad][:12]
    if frozen_examples:
        print('[Pretrain] frozen examples:', frozen_examples)

def train_one_epoch_pretrain(model, loader, optimizer, pad_idx, beta):
    model.train()
    totals = {'loss': 0.0, 'recon': 0.0, 'kld': 0.0, 'steps': 0}
    for batch in loader:
        sm_enc, sm_dec_in, sm_dec_out, prop = [t.to(device) for t in batch]
        enc_mask = (sm_enc == pad_idx)
        dec_mask = (sm_dec_in == pad_idx)
        optimizer.zero_grad(set_to_none=True)
        logits, _, mu_q, lv_q, _ = model(sm_enc, sm_dec_in, prop.float(), enc_mask, dec_mask)
        loss, recon, kld = cvae_recon_kl_pretrain_loss(logits, sm_dec_out, mu_q, lv_q, pad_idx, beta=beta)
        loss.backward()
        optimizer.step()
        totals['loss'] += float(loss.item())
        totals['recon'] += float(recon.item())
        totals['kld'] += float(kld.item())
        totals['steps'] += 1
    n = max(1, totals['steps'])
    return {k: (v / n if k != 'steps' else v) for k, v in totals.items()}

def get_finetune_loss_fn(latent_dim=LATENT_DIM):
    return ConditionalVAELoss(
        vocab_size=ft_dataset.vocab_size,
        max_beta=1.05,
        cyc_steps=800,
        num_cycles=1,
        anneal_steps=300,
        free_bits=0.10,
        capacity_max=0.0,
        capacity_inc=0.0,
        gamma=0.8,
        prop_w=0.02,
        nce=0.003,
        sig_pen_p=0.005,
        sig_pen_q=0.008,
        imb=0.14,
        latent_dim=latent_dim,
    ).to(device)

TOKEN_LAYER_KEYS = (
    'encoder.smi_embed.weight',
    'decoder.smi_embed.weight',
    'predict.weight',
    'predict.bias',
)
GRAMMAR_BASE_PREFIXES = (
    'encoder.encoder',
    'encoder.normLayer',
    'encoder.pe',
    'decoder.decoder',
    'decoder.normLayer',
    'decoder.pe',
)
LATENT_COND_PREFIXES = (
    'input_embedding',
    'to_means',
    'to_var',
    'to_prop',
    'to_prop_z',
    'decoder.to_memory',
)

def remap_cvae_token_layers_(model, src_state, src_vocab, dst_vocab):
    copied = {k: 0 for k in TOKEN_LAYER_KEYS}
    with torch.no_grad():
        for key in TOKEN_LAYER_KEYS:
            if key not in src_state or key not in model.state_dict():
                continue
            src = src_state[key]
            dst = model.state_dict()[key]
            if key.endswith('weight') and src.dim() == 2 and dst.dim() == 2:
                for tok, dst_idx in dst_vocab.items():
                    src_idx = src_vocab.get(tok)
                    if src_idx is None or src_idx >= src.size(0) or dst_idx >= dst.size(0):
                        continue
                    dst[dst_idx].copy_(src[src_idx].to(dst.device))
                    copied[key] += 1
            elif key.endswith('bias') and src.dim() == 1 and dst.dim() == 1:
                for tok, dst_idx in dst_vocab.items():
                    src_idx = src_vocab.get(tok)
                    if src_idx is None or src_idx >= src.size(0) or dst_idx >= dst.size(0):
                        continue
                    dst[dst_idx].copy_(src[src_idx].to(dst.device))
                    copied[key] += 1
    return copied

def _allow_pretrained_key(key, transfer_scope='grammar_only'):
    if key in TOKEN_LAYER_KEYS:
        return False  # token rows are handled via remap
    if transfer_scope == 'all':
        return True
    if transfer_scope == 'grammar_only':
        if any(key.startswith(pref) for pref in LATENT_COND_PREFIXES):
            return False
        if any(key.startswith(pref) for pref in GRAMMAR_BASE_PREFIXES):
            return True
        return False
    raise ValueError(f'Unknown transfer_scope: {transfer_scope}')

def load_pretrained_cvae_with_vocab_remap(model, ckpt_path, dst_vocab, transfer_scope='grammar_only'):
    blob = torch.load(ckpt_path, map_location='cpu')
    src_state = blob.get('model_state_dict', blob.get('state_dict', blob))
    src_vocab = blob.get('vocab')
    if src_vocab is None:
        raise ValueError('Pretrain checkpoint missing vocab.')

    current = model.state_dict()
    matched, skipped = {}, []
    for k, v in src_state.items():
        if not _allow_pretrained_key(k, transfer_scope=transfer_scope):
            continue
        if k in current and tuple(current[k].shape) == tuple(v.shape):
            matched[k] = v
        else:
            skipped.append(k)
    model.load_state_dict(matched, strict=False)
    copied = remap_cvae_token_layers_(model, src_state, src_vocab, dst_vocab)
    overlap = sum(1 for tok in dst_vocab if tok in src_vocab)
    print(f'[Transfer:{transfer_scope}] base matched={len(matched)} skipped={len(skipped)}')
    print(f'[Transfer:{transfer_scope}] token overlap={overlap}/{len(dst_vocab)} copied_rows={copied}')
    if skipped:
        print('Skipped examples:', skipped[:10])
    return blob

def set_finetune_stage(model, prior, stage='warmup'):
    if stage == 'warmup':
        # Freeze grammar backbone; adapt conditional/latent path and prior first.
        set_requires_grad_by_prefix(model, freeze_prefixes=GRAMMAR_BASE_PREFIXES + ('encoder.smi_embed', 'decoder.smi_embed', 'predict'), unfreeze_all=False)
        for p in prior.parameters():
            p.requires_grad = True
    elif stage == 'full':
        for p in model.parameters():
            p.requires_grad = True
        for p in prior.parameters():
            p.requires_grad = True
    else:
        raise ValueError(stage)
    mt, mtr = _count_params(model)
    pt, ptr = _count_params(prior)
    print(f'[Finetune:{stage}] model trainable {mtr:,}/{mt:,} | prior trainable {ptr:,}/{pt:,}')

def build_finetune_optimizers(model, prior):
    prop_params = [p for p in list(model.to_prop.parameters()) + list(model.to_prop_z.parameters()) if p.requires_grad]
    prop_ids = {id(p) for p in prop_params}
    base_params = [p for p in model.parameters() if p.requires_grad and id(p) not in prop_ids]
    param_groups = []
    if base_params:
        param_groups.append({'params': base_params, 'lr': FINETUNE_LR})
    if prop_params:
        param_groups.append({'params': prop_params, 'lr': FINETUNE_LR * 0.5})
    if not param_groups:
        raise RuntimeError('No trainable model params for finetune stage.')
    optim_model = AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    prior_params = [p for p in prior.parameters() if p.requires_grad]
    optim_prior = AdamW(prior_params, lr=FINETUNE_LR_PRIOR, weight_decay=WEIGHT_DECAY)
    return optim_model, optim_prior

def train_one_epoch_finetune(model, prior, loader, loss_fn, pad_idx, seq_len, step_for_loss):
    model.train(); prior.train()
    acc = {'loss': 0.0, 'recon': 0.0, 'kld': 0.0, 'prop': 0.0, 'steps': 0}
    optim_model, optim_prior = build_finetune_optimizers(model, prior)
    for batch in loader:
        sm_enc, sm_dec_in, sm_dec_out, prop = [t.to(device) for t in batch]
        enc_mask = (sm_enc == pad_idx)
        dec_mask = (sm_dec_in == pad_idx)

        optim_model.zero_grad(set_to_none=True)
        optim_prior.zero_grad(set_to_none=True)

        output, tgt, mu_q, lv_q, tgt_z = model(sm_enc, sm_dec_in, prop.float(), enc_mask, dec_mask)
        mu_p, lv_p = prior(prop.float().squeeze(-1))

        loss, BCE, KLD, kld_raw, kld_per_token, prop_mu = loss_fn(
            output.float(),
            sm_dec_out,
            mu_q,
            lv_q,
            mu_p,
            lv_p,
            tgt,
            prop.float().squeeze(-1),
            tgt_z,
            step_for_loss,
        )
        loss.backward()
        optim_model.step()
        optim_prior.step()

        acc['loss'] += float(loss.item())
        acc['recon'] += float(BCE.item())
        acc['kld'] += float(KLD.item())
        acc['prop'] += float(prop_mu.item())
        acc['steps'] += 1

    n = max(1, acc['steps'])
    return {
        'loss': acc['loss'] / n,
        'recon_per_token': (acc['recon'] / n) / seq_len,
        'kld_per_token': (acc['kld'] / n) / seq_len,
        'prop_mse': acc['prop'] / n,
        'steps': acc['steps'],
    }


In [ ]:
# ==== Stage 1: PI1M pretraining (reconstruction pretrain, transfer as grammar_only) ====
pre_ckpt_path = PRETRAIN_DIR / 'model_weights_lstm_LSTM_CVAE_pi1m_pretrain.pth'

if LOAD_EXISTING_PRETRAIN and pre_ckpt_path.exists():
    print('Using existing pretrain checkpoint:', pre_ckpt_path)
    pretrain_history = []
else:
    pre_model = build_cvae(
        vocab_size=pi1m_ds.vocab_size,
        max_len=pi1m_ds.max_len,
        pad_idx=pi1m_ds.vocab['[PAD]'],
        latent_dim=LATENT_DIM,
    )
    pre_pad_idx = pi1m_ds.vocab['[PAD]']

    # PI1M에는 전도도 라벨이 없으므로 property regression heads / input_embedding은 pretrain 대상에서 제외.
    # NOTE: to_means/to_var/decoder.to_memory는 reconstruction 경로에 직접 필요하므로 trainable 유지.
    pretrain_freeze_non_target_heads(pre_model, freeze_prefixes=PRETRAIN_FREEZE_PREFIXES)
    pre_optim = AdamW([p for p in pre_model.parameters() if p.requires_grad], lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)

    pretrain_history = []
    for ep in tqdm(range(1, PRETRAIN_EPOCHS + 1), desc='PI1M pretrain epochs'):
        t0 = time.time()
        stats = train_one_epoch_pretrain(pre_model, pi1m_loader, pre_optim, pre_pad_idx, PRETRAIN_KL_BETA)
        stats['epoch'] = ep
        stats['minutes'] = (time.time() - t0) / 60.0
        pretrain_history.append(stats)
        print(f"[Pretrain][{ep}/{PRETRAIN_EPOCHS}] loss={stats['loss']:.5f} recon={stats['recon']:.5f} kld={stats['kld']:.5f} ({stats['minutes']:.2f} min)")

    torch.save({
        'model_state_dict': pre_model.state_dict(),
        'vocab': pi1m_ds.vocab,
        'vocab_size': pi1m_ds.vocab_size,
        'max_len': pi1m_ds.max_len,
        'latent_dim': LATENT_DIM,
        'pi1m_csv': str(PI1M_CSV),
        'pi1m_max_rows': PI1M_MAX_ROWS,
        'pi1m_property_value': 0.0,
        'pretrain_epochs': PRETRAIN_EPOCHS,
        'pretrain_kl_beta': PRETRAIN_KL_BETA,
        'pretrain_freeze_prefixes': PRETRAIN_FREEZE_PREFIXES,
        'pretrain_transfer_scope_recommended': PRETRAIN_TRANSFER_SCOPE,
        'pretrain_history': pretrain_history,
    }, pre_ckpt_path)
    print('Saved pretrain checkpoint ->', pre_ckpt_path)


In [ ]:
# ==== Stage 2: finetune on LSTM_CVAE dataset (scalar conductivity retained, 2-stage) ====
ft_model = build_cvae(
    vocab_size=ft_dataset.vocab_size,
    max_len=ft_dataset.max_len,
    pad_idx=ft_dataset.vocab['[PAD]'],
    latent_dim=LATENT_DIM,
)
ft_prior = build_prior(max_len=ft_dataset.max_len, latent_dim=LATENT_DIM)

if pre_ckpt_path.exists():
    _ = load_pretrained_cvae_with_vocab_remap(
        ft_model,
        pre_ckpt_path,
        ft_dataset.vocab,
        transfer_scope=PRETRAIN_TRANSFER_SCOPE,
    )
else:
    print('Pretrain checkpoint not found. Finetuning from scratch.')

loss_fn = get_finetune_loss_fn(LATENT_DIM)
PAD_IDX = ft_dataset.vocab['[PAD]']
seq_len = ft_dataset.max_len + 2

# Property heads는 scratch 적응이 빠르도록 재초기화 유지
for head in [ft_model.to_prop, ft_model.to_prop_z]:
    nn.init.xavier_uniform_(head.weight)
    nn.init.zeros_(head.bias)

finetune_history = []
stage_plan = []
if FINETUNE_STAGE1_EPOCHS > 0:
    stage_plan.append(('warmup', FINETUNE_STAGE1_EPOCHS))
if FINETUNE_STAGE2_EPOCHS > 0:
    stage_plan.append(('full', FINETUNE_STAGE2_EPOCHS))

if RUN_FINETUNE:
    global_epoch = 0
    for stage_name, n_epochs in stage_plan:
        print(f'\n===== Finetune stage: {stage_name} ({n_epochs} epochs) =====')
        set_finetune_stage(ft_model, ft_prior, stage=stage_name)
        for _ in tqdm(range(n_epochs), desc=f'Finetune:{stage_name}'):
            global_epoch += 1
            t0 = time.time()
            row = train_one_epoch_finetune(
                ft_model,
                ft_prior,
                train_dataloader,
                loss_fn,
                PAD_IDX,
                seq_len,
                step_for_loss=global_epoch,
            )
            row['epoch'] = global_epoch
            row['stage'] = stage_name
            row['sec'] = time.time() - t0
            finetune_history.append(row)
            print(
                f"[Finetune:{stage_name}][{global_epoch}/{FINETUNE_EPOCHS}] "
                f"loss={row['loss']:.5f} recon/tok={row['recon_per_token']:.5f} "
                f"kld/tok={row['kld_per_token']:.5f} prop_mse={row['prop_mse']:.5f} ({row['sec']:.1f}s)"
            )
else:
    print('RUN_FINETUNE=False -> skip finetuning loop')


In [ ]:
# ==== Save finetuned weights for generation / FCD ====
model_save_path = WEIGHTS_DIR / 'model_weights_lstm_LSTM_CVAE_pi1m_finetuned.pth'
prior_save_path = WEIGHTS_DIR / 'model_weights_prior_lstm_LSTM_CVAE_pi1m_finetuned.pth'
bundle_path = WEIGHTS_DIR / 'model_weights_lstm_LSTM_CVAE_pi1m_finetuned_bundle.pth'

torch.save(ft_model.state_dict(), model_save_path)
torch.save(ft_prior.state_dict(), prior_save_path)
torch.save({
    'model_state_dict': ft_model.state_dict(),
    'prior_state_dict': ft_prior.state_dict(),
    'vocab': ft_dataset.vocab,
    'max_len': ft_dataset.max_len,
    'latent_dim': LATENT_DIM,
    'pretrain_ckpt': str(pre_ckpt_path),
    'pretrain_transfer_scope': PRETRAIN_TRANSFER_SCOPE,
    'pretrain_freeze_prefixes': PRETRAIN_FREEZE_PREFIXES,
    'finetune_stage1_epochs': FINETUNE_STAGE1_EPOCHS,
    'finetune_stage2_epochs': FINETUNE_STAGE2_EPOCHS,
    'pretrain_history': pretrain_history,
    'finetune_history': finetune_history,
}, bundle_path)

print('Saved model ->', model_save_path)
print('Saved prior ->', prior_save_path)
print('Saved bundle ->', bundle_path)
print('Last pretrain history:', pretrain_history[-1] if pretrain_history else None)
print('Last finetune history:', finetune_history[-1] if finetune_history else None)


## Notes

- PI1M full 99만개는 dense tensor 캐시가 커질 수 있다. 첫 실행은 `PI1M_MAX_ROWS=100000~300000`으로 테스트 후 늘리는 것을 권장.
- pretrain은 scalar 경로를 유지하되, PI1M에 라벨이 없으므로 `property_value=0.0` 상수로 넣고 reconstruction+KL 중심으로 진행한다.
- finetune 단계에서 기존 `TransCVAE_PSMILES` scalar conductivity loss/ prior 학습을 그대로 사용한다.
